# Initial Setup

In [16]:
%pip install catboost lightgbm optuna holidays duckdb

## Imports

In [17]:
import numpy as np
import pandas as pd
import duckdb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

## Variables

In [18]:
def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

DRIVE_FOLDER_NAME = "Store Sales"

In [19]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [20]:
TARGET = "sales"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']


## Helper Functions

### Data Loader

In [21]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Loader of additional data

In [22]:
def load_helper_data(option: str="local"):
  '''
  returns oil_df, transactions_df, holidays_events_df, stores_df

  '''
  if option == "local":
    oil_df = pd.read_csv('oil.csv')
    transactions_df = pd.read_csv('transactions.csv')
    holidays_events_df = pd.read_csv('holidays_events.csv')
    stores_df = pd.read_csv('stores.csv')


  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    oil_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/oil.csv')
    transactions_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/transactions.csv')
    holidays_events_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/holidays_events.csv')
    stores_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/stores.csv')
    
  oil_df['date'] =  pd.to_datetime(oil_df['date'])
  transactions_df['date'] =  pd.to_datetime(transactions_df['date'])
  holidays_events_df['date'] =  pd.to_datetime(holidays_events_df['date'])
  


  return oil_df, transactions_df, holidays_events_df, stores_df

### Data Splitter

In [23]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

## Create DataFrames

In [24]:
train_df, test_df, df = load_data(ENVIRONMENT)
oil_df, transactions_df, holidays_events_df, stores_df = load_helper_data(ENVIRONMENT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Tables infos

## Overall Info

### Main DF

In [0]:
df.head()

In [0]:
df.groupby('source').agg(min_date=('date', 'min'), max_date=('date', 'max'))

In [0]:
df.describe(include='all')

### Stores

In [0]:
stores_df.head()

In [0]:
stores_df.describe(include='all')

### Transactions

In [0]:
transactions_df.head()

In [0]:
transactions_df.describe(include='all')

### Holidays

In [0]:
holidays_events_df.head()

In [0]:
holidays_events_df.describe()

### Oil

In [0]:
oil_df.head()

In [0]:
oil_df.describe()

In [0]:
df.head()

# Merging datasets

In [25]:
def generate_feature_table(
    df: pd.DataFrame,
    oil_df: pd.DataFrame = oil_df,
    holidays_events_df: pd.DataFrame = holidays_events_df,
    transactions_df: pd.DataFrame = transactions_df,
    stores_df: pd.DataFrame = stores_df,
) -> pd.DataFrame:
    
    # Legit holidays masks
    is_transferred = holidays_events_df['transferred']
    legit_holidays = holidays_events_df['type'].isin(['Holiday', 'Transfer', 'Bridge', 'Additional'])
    
    # Flags for different type of holidays/events
    is_workday = holidays_events_df['type']  == 'Work Day'
    is_event = holidays_events_df['type']  == 'Event'
    is_national = holidays_events_df['locale'] == 'National'
    is_regional = holidays_events_df['locale'] == 'Regional'
    is_local = holidays_events_df['locale'] == 'Local'

    # Generate DataFrames per holiday type, filtered to legit holidays
    national_holidays = holidays_events_df.loc[(~is_transferred) & (is_national) & (legit_holidays)]
    regional_holidays = holidays_events_df.loc[(~is_transferred) & (is_regional) & (legit_holidays)]
    local_holidays = holidays_events_df.loc[(~is_transferred) & (is_local) & (legit_holidays)]

    # Deduplicate rows and add additional features
    national_holidays = national_holidays.groupby('date').agg(nat_num_holidays=('type', 'count'), nat_holiday_types =('type', ', '.join)).sort_values(by='nat_num_holidays', ascending=False).reset_index()
    national_holidays['is_national'] = 1

    regional_holidays = regional_holidays.groupby(['date', 'locale_name']).agg(reg_num_holidays=('type', 'count'), reg_holiday_types =('type', ', '.join)).sort_values(by='reg_num_holidays', ascending=False).reset_index()
    regional_holidays['is_regional'] = 1

    local_holidays = local_holidays.groupby(['date', 'locale_name']).agg(loc_num_holidays=('type', 'count'), local_holiday_types =('type', ', '.join)).sort_values(by='loc_num_holidays', ascending=False).reset_index()
    local_holidays['is_local'] = 1

    # Generate 2 DataFrames for workdays and events
    workdays = holidays_events_df.loc[(~is_transferred) & (is_workday)].copy()
    workdays['is_workday'] = 1
    workdays = workdays[['date', 'is_workday']].drop_duplicates(subset=['date'], keep='first')

    events = holidays_events_df.loc[(~is_transferred) & (is_event)].copy()
    events['is_event'] = 1
    events = events[['date', 'is_event']].drop_duplicates(subset=['date'], keep='first')

    # Fill weekend oil prices with previous available price
    oil_df_filled = (oil_df.set_index("date")
                    .reindex(pd.date_range(oil_df["date"].min(), df["date"].max()))
                    .ffill().bfill()
                    .rename_axis("date").reset_index())

    df = (df.merge(transactions_df, on=['date', 'store_nbr'], how='left')
                .merge(stores_df, on='store_nbr', how='left')
                .merge(oil_df_filled, on='date', how='left')
                .merge(national_holidays, on='date', how='left')
                .merge(regional_holidays, left_on=['date', 'state'], right_on=['date','locale_name'], how='left').drop(columns=['locale_name'])
                .merge(local_holidays, left_on=['date','city'], right_on=['date','locale_name'], how='left').drop(columns=['locale_name'])
                .merge(workdays, on='date', how='left')
                            .merge(events, on='date', how='left')
                )

    # Adjust holidays and transactions NULLs
    is_train = df["sales"].notna()

    df["store_sales"] = df.groupby(["date", "store_nbr"])["sales"].transform("sum")

    transaction_null = df["transactions"].isna()
    store_closed = df["store_sales"] == 0

    df.loc[is_train & transaction_null & store_closed, "transactions"] = 0
    num_holiday_cols = ['is_national', 'is_regional', 'is_local', 'is_workday', 'is_event', 'nat_num_holidays', 'reg_num_holidays', 'loc_num_holidays']
    str_holiday_cols = ['nat_holiday_types', 'reg_holiday_types', 'local_holiday_types']

    df[num_holiday_cols] = df[num_holiday_cols].fillna(0).astype(int)
    df[str_holiday_cols] = df[str_holiday_cols].fillna('none')

    return df

In [26]:
df = generate_feature_table(df, oil_df=oil_df, holidays_events_df=holidays_events_df, transactions_df=transactions_df, stores_df=stores_df)

In [27]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,source,transactions,city,state,...,is_national,reg_num_holidays,reg_holiday_types,is_regional,loc_num_holidays,local_holiday_types,is_local,is_workday,is_event,store_sales
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train,0.0,Quito,Pichincha,...,1,0,none,0,0,none,0,0,0,0.0
1,1,2013-01-01,1,BABY CARE,0.0,0,train,0.0,Quito,Pichincha,...,1,0,none,0,0,none,0,0,0,0.0
2,2,2013-01-01,1,BEAUTY,0.0,0,train,0.0,Quito,Pichincha,...,1,0,none,0,0,none,0,0,0,0.0
3,3,2013-01-01,1,BEVERAGES,0.0,0,train,0.0,Quito,Pichincha,...,1,0,none,0,0,none,0,0,0,0.0
4,4,2013-01-01,1,BOOKS,0.0,0,train,0.0,Quito,Pichincha,...,1,0,none,0,0,none,0,0,0,0.0


# EDA

In [0]:
df.info()

In [0]:
df.describe(include='object').T

In [0]:
df.describe(include='number').T

In [0]:
df.columns

## Store Nbr

In [0]:
plt.figure(figsize=(15,5))
ax1 = sns.barplot(
    df,
    x='store_nbr',
    y='store_sales',
    estimator='sum',
    errorbar=None,
)

ax2 = ax1.twinx()

sns.pointplot(
    data=df,
    x='store_nbr',
    y='store_sales',
    estimator='mean',
    ax=ax2,
    color='red',
    markers='x',
    linestyles='-',
    label="Mean Daily store_sales",
)


sns.pointplot(
    data=df,
    x='store_nbr',
    y='store_sales',
    estimator='median',
    ax=ax2,
    color='orange',
    markers='*',
    linestyles='--',
    label = "Median Daily store_sales",
)

### Store Opening Times

In [0]:
plt.figure(figsize=(8, 5))

first_open_date = df.loc[df['sales'] > 0].groupby('store_nbr')['date'].min().reset_index()

fig = px.scatter(
    first_open_date,
    x='date',
    y='store_nbr'
)

fig.show()

### Sum / medians of sales per store

In [0]:
first_open_date = df.loc[df['sales'] > 0].groupby('store_nbr')['date'].agg(min_date="min").reset_index()
plot_df = df.copy()

plot_df = plot_df.merge(first_open_date, on='store_nbr', how='left')
plot_df = plot_df.loc[plot_df['date'] >= plot_df['min_date']]

col, metric = ["sales", "store_sales"], ["sum", "median"]

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))

for i, c in enumerate(col):         # i -> row
    for j, m in enumerate(metric): # j - column
        order = plot_df.groupby("store_nbr")[c].agg(m).sort_values().index
        plot_ax = axs[i,j]
        sns.barplot(
            plot_df,
            x="store_nbr",
            y=c,
            estimator=m,
            errorbar=None,
            ax=plot_ax,
            order=order,
            hue='type'
            
        )
        plot_ax.set_title(f"{m.capitalize()} of {c.capitalize()} per store")
        plot_ax.tick_params(axis="x", rotation=270)
plt.tight_layout()

In [0]:
col, metric = ["sales", "store_nbr"], ["sum", "median"]

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
for i, c in enumerate(col):         # i -> row
    for j, m in enumerate(metric): # j - column
        order = df.groupby("family")[c].agg(m).sort_values().index
        plot_ax = axs[i,j]
        sns.barplot(
            df,
            x="family",
            y=c,
            estimator=m,
            errorbar=None,
            ax=plot_ax,
            order=order,
            
        )
        plot_ax.set_title(f"{m.capitalize()} of {c.capitalize()} per store")
        plot_ax.tick_params(axis="x", rotation=270)
plt.tight_layout()

In [0]:
store_family_summary = df.groupby(["store_nbr", "family"]).agg(
    sum_sales=("sales", "sum"),
    median_sales=("sales", "median"),
    average_sales=("sales", "mean"),
).reset_index().pivot(
    index='family',
    columns='store_nbr',
    values=['sum_sales', 'median_sales', 'average_sales']
)

fig, axs = plt.subplots(3,1,figsize=(15,15))
values=['sum_sales', 'median_sales', 'average_sales']

for i, v in enumerate(values):
	order = store_family_summary[v].sum(axis=1).sort_values(ascending=False).index
	heatmap_df = store_family_summary[v].loc[order]

	sns.heatmap(
			heatmap_df,
			# annot=True,
			fmt=f",.0f",
			cmap="YlGnBu",
			linewidths=0.2,
			annot_kws={"size": 7},
			# square=True,
			robust=True,
			# cbar_kws={"shrink": 1, "aspect": 30, "pad": 0.01},
			ax=axs[i]
	)

	axs[i].xaxis.tick_top()
	axs[i].xaxis.set_label_position('top')
	axs[i].set_title(f"family by {v} per store_nbr")

plt.tight_layout()

In [0]:
store_family_summary = df.groupby(["type", "family"]).agg(
    sum_sales=("sales", "sum"),
    median_sales=("sales", "median"),
    average_sales=("sales", "mean"),
).reset_index().pivot(
    index='family',
    columns='type',
    values=['sum_sales', 'median_sales', 'average_sales']
)

fig, axs = plt.subplots(3,1,figsize=(15,15))
values=['sum_sales', 'median_sales', 'average_sales']

for i, v in enumerate(values):
	order = store_family_summary[v].sum(axis=1).sort_values(ascending=False).index
	heatmap_df = store_family_summary[v].loc[order]

	sns.heatmap(
			heatmap_df,
			# annot=True,
			fmt=f",.0f",
			cmap="YlGnBu",
			linewidths=0.2,
			annot_kws={"size": 7},
			# square=True,
			robust=True,
			# cbar_kws={"shrink": 1, "aspect": 30, "pad": 0.01},
			ax=axs[i]
	)

	axs[i].xaxis.tick_top()
	axs[i].xaxis.set_label_position('top')
	axs[i].set_title(f"family by {v} per type")

plt.tight_layout()

In [0]:
store_family_summary = df.groupby(["cluster", "family"]).agg(
    sum_sales=("sales", "sum"),
    median_sales=("sales", "median"),
    average_sales=("sales", "mean"),
).reset_index().pivot(
    index='family',
    columns='cluster',
    values=['sum_sales', 'median_sales', 'average_sales']
)

fig, axs = plt.subplots(3,1,figsize=(15,15))
values=['sum_sales', 'median_sales', 'average_sales']

for i, v in enumerate(values):
	order = store_family_summary[v].sum(axis=1).sort_values(ascending=False).index
	heatmap_df = store_family_summary[v].loc[order]

	sns.heatmap(
			heatmap_df,
			# annot=True,
			fmt=f",.0f",
			cmap="YlGnBu",
			linewidths=0.2,
			annot_kws={"size": 7},
			# square=True,
			robust=True,
			# cbar_kws={"shrink": 1, "aspect": 30, "pad": 0.01},
			ax=axs[i]
	)

	axs[i].xaxis.tick_top()
	axs[i].xaxis.set_label_position('top')
	axs[i].set_title(f"family by {v} per cluster")

plt.tight_layout()

# Feature Engineering

In [ ]:
def feature_engineering(fdf: pd.DataFrame = df) -> pd.DataFrame:
    df = fdf.copy()
    # Date Transformations
    d = df['date'].dt
    df['year']         = d.year
    df['month']        = d.month            # 1–12
    df['day']          = d.day              # day of month, 1–31
    df['dayofweek']    = d.dayofweek        # Mon=0 … Sun=6
    df['dayofyear']    = d.dayofyear        # 1–366
    df['quarter']      = d.quarter          # 1–4
    df['weekofyear']   = d.isocalendar().week.astype(int)   # 1–53
    df['days_in_month'] = d.days_in_month
    df['is_weekend']       = (d.dayofweek >= 5).astype(int)
    df['is_month_start']   = d.is_month_start.astype(int)
    df['is_month_end']     = d.is_month_end.astype(int)
    df['is_quarter_start'] = d.is_quarter_start.astype(int)
    df['is_quarter_end']   = d.is_quarter_end.astype(int)
    df['is_year_start']    = d.is_year_start.astype(int)
    df['is_year_end']      = d.is_year_end.astype(int)

    mask = (df['is_regional'] == 1) | (df['is_local'] == 1) | (df['is_national'] == 1)
    df['is_holiday'] = mask.astype(int)

    #! TODO: days before / after holiday
    #! TODO: days after event
    #! TODO: store sales to transactions ratio --- DROP ACTUAL VALUES USE ONLY LAG
    #! TODO: add share of sold products out of all products --- DROP ACTUAL VALUES USE ONLY LAG
    #! TODO: share of sales per city / state by each --- DROP ACTUAL VALUES USE ONLY LAG
    #! TODO: impute missing transactions
    #! TODO: remove stores before their opening date

    return df

In [38]:
temp_df = feature_engineering(df)
temp_df.head()

,id,date,store_nbr,family,sales,onpromotion,source,transactions,city,state,...,weekofyear,days_in_month,is_weekend,is_month_start,is_month_end,is_quarter_start,is_quarter_end,is_year_start,is_year_end,is_holiday
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train,0.0,Quito,Pichincha,...,1,31,0,1,0,1,0,1,0,1
1,1,2013-01-01,1,BABY CARE,0.0,0,train,0.0,Quito,Pichincha,...,1,31,0,1,0,1,0,1,0,1
2,2,2013-01-01,1,BEAUTY,0.0,0,train,0.0,Quito,Pichincha,...,1,31,0,1,0,1,0,1,0,1
3,3,2013-01-01,1,BEVERAGES,0.0,0,train,0.0,Quito,Pichincha,...,1,31,0,1,0,1,0,1,0,1
4,4,2013-01-01,1,BOOKS,0.0,0,train,0.0,Quito,Pichincha,...,1,31,0,1,0,1,0,1,0,1


In [0]:
# TODO: create features such as meadian per store and nunique stores per family but only during cross validation to prevent leakage

# Quick Test

In [66]:
X, y, submission_df = data_splitter(temp_df)

dates = np.sort(X['date'].unique())

tscv = TimeSeriesSplit(
    n_splits=5,
    test_size=16
)

for fold, (tr_date_idx, val_date_idx) in enumerate(tscv.split(dates)):
    tr_dates = dates[tr_date_idx]
    val_dates = dates[val_date_idx]

    tr_idx = np.where(X['date'].isin(tr_dates))[0]
    val_idx = np.where(X['date'].isin(val_dates))[0]


    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    print(fold, X_train['date'].max(), X_val['date'].min(), X_val['date'].nunique())

0 2017-05-27 00:00:00 2017-05-28 00:00:00 16
1 2017-06-12 00:00:00 2017-06-13 00:00:00 16
2 2017-06-28 00:00:00 2017-06-29 00:00:00 16
3 2017-07-14 00:00:00 2017-07-15 00:00:00 16
4 2017-07-30 00:00:00 2017-07-31 00:00:00 16
